# 19. Exception Handling (5+ Years Interview Guide)
Deep dive into Python exception hierarchy, try-except-else-finally lifecycle, exception chaining (raise from), custom enterprise error hierarchies, and Python 3.11+ ExceptionGroups.

### Key 5-Year Interview Concepts Covered:
- **Exception Hierarchy**: Why catching `Exception` is safe while catching `BaseException` intercepting `KeyboardInterrupt`/`SystemExit` is an antipattern.
- **Control Flow Hooks**: Complete lifecycle of `try` -> `except` -> `else` (success) -> `finally` (guaranteed cleanup).
- **Exception Chaining (`raise from`)**: Preserving root causes via `__cause__` and `__context__` for distributed tracing.
- **Python 3.11+ Exception Groups**: Handling concurrent multi-task failures using `ExceptionGroup` and `except*`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Basic Exception Handling (`try-except`)
**Explanation**: The `try-except` block intercepts runtime errors (`Exceptions`) without terminating the Python process. When an error occurs in the `try` block, CPython halts execution of that block and searches for matching `except` handler clauses.

**Syntax**: `try: risky_operation()
except ValueError as err: handle_error(err)`

In [ ]:
try: int('abc')
except ValueError as error_message: print('Caught:', error_message)

### 2. Catching Multiple Specific Exceptions
**Explanation**: Handle distinct exception types with separate `except` clauses to apply tailored recovery strategies for each failure mode (e.g. retrying a network timeout vs logging an invalid payload).

**Syntax**: `except ConnectionError: retry()
except ValueError: log_bad_payload()`

In [ ]:
try: 1/0
except ValueError: print()
except ZeroDivisionError: print('Div by Zero')

### 3. Grouping Exception Tuples
**Explanation**: Multiple exception types that share identical recovery logic can be caught together in a tuple: `except (TypeError, ValueError, KeyError) as e:`. Parentheses are syntactically required; omitting them causes Python to treat the second type as the variable name.

**Syntax**: `except (ValueError, KeyError) as err: ...`

In [ ]:
try: int([])
except (ValueError, TypeError) as error_message: print('Grouped error caught:', error_message)

### 4. Exception Hierarchy: `Exception` vs `BaseException`
**Explanation**: In Python, all standard user errors inherit from `Exception`. `BaseException` is the root of the entire hierarchy and includes system-exiting signals (`KeyboardInterrupt`, `SystemExit`, `GeneratorExit`). Catching bare `except:` or `except BaseException:` prevents users from stopping scripts with Ctrl+C and prevents clean worker shutdowns.

**Syntax**: `except Exception as e:  # Safe catch-all` / `except BaseException:  # Antipattern`

In [ ]:
try: int([])
except Exception as error_instance: print('Generic catch:', type(error_instance))

### 5. The Success Hook (`else`)
**Explanation**: The `else` clause executes ONLY if the `try` block completes successfully without raising any exceptions. Placing code in `else` rather than inside `try` prevents accidentally catching unintended exceptions raised by subsequent code.

**Syntax**: `try: data = parse() 
except ParseError: handle() 
else: save_to_db(data)`

In [ ]:
try: execution_value = 1
except: pass
else: print('Success path executed')

### 6. The Cleanup Hook (`finally`)
**Explanation**: The `finally` block ALWAYS executes before leaving the `try` statement, regardless of whether exceptions were raised, caught, or unhandled—even if the `try` or `except` blocks execute an explicit `return`! It is reserved for releasing unmanaged resources.

**Syntax**: `try: lock.acquire() 
finally: lock.release()  # Guaranteed execution`

In [ ]:
try: 1/0
except: pass
finally: print('Unconditional cleanup block executed')

### 7. Raising Exceptions Explicitly (`raise`)
**Explanation**: Use `raise ExceptionClass('message')` to signal invalid states or failed preconditions. Always provide actionable, detailed error messages containing the invalid input values to aid debugging.

**Syntax**: `if amount < 0: raise ValueError(f'Amount cannot be negative: {amount}')`

In [ ]:
try: raise ValueError('Manual error')
except ValueError as error_message: print(error_message)

### 8. Custom Exception Hierarchies
**Explanation**: Enterprise applications define domain-specific exception hierarchies inheriting from `Exception`. Creating a base `FintechError` and subclasses (`PaymentError`, `InsufficientFundsError`, `FraudError`) allows library consumers to catch either specific errors or all module errors cleanly.

**Syntax**: `class PaymentError(Exception): pass` / `class FraudError(PaymentError): pass`

In [ ]:
class CustomAuditException(Exception): pass
print(CustomAuditException)

### 9. Exception Propagation & Call Stack Unwinding
**Explanation**: If an exception is not caught in the active function frame, CPython unwinds the call stack upward through calling functions until a matching handler is found. If it reaches the top-level module unhandled, CPython prints the traceback and exits with non-zero exit code.

**Syntax**: `# Unwinds stack frame by frame until handled`

In [ ]:
def raise_error(): raise ValueError()
def call_raiser(): raise_error()
try: call_raiser()
except ValueError: print('Caught propagated error')

### 10. Exception Chaining (`raise ... from ...`)
**Explanation**: Introduced in PEP 3134, `raise NewException(...) from original_exc` explicitly chains exceptions, setting `__cause__` and displaying both tracebacks in logs. This preserves root-cause visibility when wrapping low-level errors (like `psycopg2.OperationalError`) into domain errors (`DatabaseConnectionError`). To intentionally suppress root cause, use `raise NewException() from None`.

**Syntax**: `raise PaymentProcessingError('Failed') from original_error`

In [ ]:
try:
    try: 1/0
    except ZeroDivisionError as error_instance:
        raise ValueError('Chained') from error_instance
except ValueError as chained_error:
    print('Chained:', chained_error, 'Original:', repr(chained_error.__cause__))

### 11. Accessing Traceback Attributes (`__traceback__`)
**Explanation**: Active exceptions carry a `__traceback__` object containing the execution stack frames. The standard `traceback` module formats or extracts stack frames for structured telemetry reporting.

**Syntax**: `import traceback; tb_str = traceback.format_exc()`

In [ ]:
try: 1/0
except ZeroDivisionError as error_instance:
    print('Traceback active object:', error_instance.__traceback__)

### 12. Re-Raising Active Exceptions (`raise`)
**Explanation**: A bare `raise` inside an `except` block re-raises the currently active exception while preserving its original traceback and stack history. Use this when logging an error or rolling back a transaction before letting the error propagate upward.

**Syntax**: `except Exception: log.error('Failed'); raise  # Preserves traceback`

In [ ]:
try:
    try: 1/0
    except:
        print('Log locally'); raise
except:
    print('Reraised and caught')

### 13. Accessing Exception Arguments (`args`)
**Explanation**: Exceptions store passed constructor parameters in their `.args` tuple. Accessing `err.args` allows extracting structured error codes or metadata attached by custom exception classes.

**Syntax**: `error_code = err.args[0]`

In [ ]:
try: raise ValueError('err', 404)
except ValueError as error_instance: print('Args:', error_instance.args)

### 14. Exception Groups & `except*` (Python 3.11+ PEP 654)
**Explanation**: When concurrent tasks (like `asyncio.TaskGroup`) fail simultaneously, multiple distinct exceptions can occur. Python 3.11 introduced `ExceptionGroup` and the `except*` syntax to match and handle specific sub-types of concurrent exceptions independently.

**Syntax**: `try: ...
except* ValueError as eg: ...
except* ConnectionError as eg: ...`

In [ ]:
# ExceptionGroups structure validation
print('ExceptionGroup loaded safely')

### 15. The `warnings` Module vs Exceptions
**Explanation**: Use `warnings.warn('Deprecated feature', DeprecationWarning)` for non-fatal notifications, deprecation warnings, or performance advice that should alert developers without halting program execution.

**Syntax**: `import warnings; warnings.warn('Use v2 API', DeprecationWarning)`

In [ ]:
import warnings
warnings.warn('Non-fatal system alert')

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Building resilient payment pipeline error hierarchies, preserving audit trails with exception chaining, and testing exception recovery.


In [ ]:
# Solution:
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(10):
        row = f.readline().strip().split(',')
        try:
            amount = float(row[3])
        except (ValueError, IndexError) as e:
            print('Gracefully caught invalid parsing item:', e)


### Q2: Exception Chaining for Fraud Detection Audits
**Explanation**: **Scenario**: Parse transaction rows, intercept formatting errors, and re-raise them chained to a custom domain exception `FraudDetectionError` using `raise ... from ...`.

**Syntax**: `raise FraudDetectionError('Fraud detected in ledger') from parse_error`

In [ ]:
# Solution:
class FraudDetectionError(Exception): pass
try:
    try: raise ValueError('Missing transaction data field')
    except ValueError as e:
        raise FraudDetectionError('Suspicious log detected') from e
except FraudDetectionError as e:
    print('Fraud Exception:', e, 'Cause:', repr(e.__cause__))
